# Tutorial 06 - Convolutional Neural Networks

In this notebook, you will build and compare image classifiers for the cats-vs-dogs dataset.

## Learning goals
- understand why fully connected networks are not ideal for image data,
- implement a simple convolutional neural network in PyTorch,
- compare models by validation performance and by number of trainable parameters,
- visualize learned filters from a pretrained network and interpret what early convolution layers detect.

This notebook is an **exercise notebook**: several key code cells are intentionally incomplete and must be filled in by you.

# Download the Dataset
We are using the cats-vs-dogs dataset from previous tutorials. The dataset can be downloaded from:
https://www.microsoft.com/en-us/download/details.aspx?id=54765

**Data:** labeled images of cats and dogs.

**Goal:** learn a **classifier**, i.e. a function that maps an input image to one of the two labels: `cat` or `dog`.

## What will happen in this notebook?
1. Load and preprocess the image dataset.
2. Train a multilayer perceptron (MLP) as a baseline.
3. Implement and train convolutional neural networks.
4. Compare the models in terms of accuracy and parameter count.
5. Visualize first-layer filters of a pretrained AlexNet model.

Run the notebook from top to bottom. Whenever you see `TODO`, complete the missing code before continuing.

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
import torch
import torch.nn as nn
import PIL
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("We are using {}".format(device))

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import Resize, ToTensor, Normalize, Compose
root_dir = '../Tutorial 01/PetImages'

target_size = (32, 32)
transforms = Compose([Resize(target_size), # Resizes image
                    ToTensor(),           # Converts to Tensor, scales to [0, 1] float (from [0, 255] int)
                    Normalize(mean=(0.5, 0.5, 0.5,), std=(0.5, 0.5, 0.5)), # scales to [-1.0, 1.0]
                    ])
corrupted_files = ["../Tutorial 01/PetImages/Cat/666.jpg", "../Tutorial 01/PetImages/Dog/11702.jpg"]
dataset_ = ImageFolder(
    root_dir, 
    transform=Resize((40,40)), 
    is_valid_file=lambda x: x.endswith("jpg") and x not in corrupted_files)

In [ ]:
class RAMDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        data = []
        for sample in tqdm(dataset):
            data.append(sample)
        self.n = len(data)
        self.data = data
        self.transform = transform
        
    def __getitem__(self, ind):
        if self.transform is not None:
            return self.transform(self.data[ind][0]), self.data[ind][1]
        else:
            return self.data[ind]
    
    def set_tranform(self, transform):
        self.transform = transform
    
    def __len__(self):
        return self.n

In [ ]:
dataset = RAMDatasetWrapper(dataset_, transform=transforms)

In [ ]:
train_set, val_set = torch.utils.data.random_split(dataset, [22000, 2998])

In [ ]:
from torch.utils.data import DataLoader
batch_size = 64

train_dataloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)

# Baseline - MLP model

Before using convolutions, we first train a standard multilayer perceptron (MLP) on the image data.

## Why do this?
This gives us a **baseline**. Later, we can compare the MLP and the ConvNet in terms of:
- validation accuracy,
- training behavior,
- number of parameters.

Recall that an MLP does **not** explicitly use the 2D spatial structure of the image. It only sees a flattened vector of pixel values.

In [ ]:
from utils_train import fit, plot

In [ ]:
class MLPModel(nn.Module):
    
    def __init__(self, input_dim, hidden_dim):
        super(MLPModel, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )
    
    def forward(self, input):
        input = input.view(input.size(0), -1)
        return self.layers(input)

In [ ]:
# TODO: Set up the MLP baseline for training.
#
# Create the model, move it to the selected device, and choose optimizer,
# loss function, and training hyperparameters.

In [ ]:
# TODO: Train the MLP and visualize its learning curves.
#
# Store the returned training history so you can compare it later to the
# convolutional models.

### Reference result
With some regularization, the best accuracy obtained previously for this MLP model was about **71%**.

Use this only as a rough orientation, not as a strict target. Small deviations are normal.

## Convolutional Neural Network

Now you will implement a convolutional neural network (ConvNet).

## Why convolutions?
Convolutional layers exploit the spatial structure of images:
- nearby pixels are related,
- the same local pattern can appear in different parts of the image,
- parameter sharing makes the model much more efficient.

## Architecture for the first ConvNet
Use the following structure:
- input: `3 x 32 x 32`
- `Conv2d(3, 16, kernel_size=5)`
- `ReLU`
- `MaxPool2d(kernel_size=2, stride=2)`
- `Conv2d(16, 32, kernel_size=5)`
- `ReLU`
- `MaxPool2d(kernel_size=2, stride=2)`
- flatten
- linear layer to 2 output logits

Your job is to implement the model and train it.

In [ ]:
class ConvModel(nn.Module):

    def __init__(self):
        super(ConvModel, self).__init__()

        # TODO: implement the convolutional network described above.
        # Build a feature extractor and a final classifier.
        self.conv_layers = nn.Sequential(
            # ...
        )

        # TODO: define the classifier that maps the extracted features to 2 classes.
        # self.linear_layer = ...

    def forward(self, input):
        # TODO: apply the network and return class logits.
        raise NotImplementedError("Implement the ConvModel forward pass")

In [ ]:
# TODO: Set up `ConvModel` for training.
#
# Use a training configuration that makes a fair comparison with the MLP baseline.

In [ ]:
# TODO: Train `ConvModel` and plot its learning curves.
#
# Compare its behavior to the MLP once training is complete.

### Exercise: Build a second ConvNet with average pooling

In the first ConvNet, the classifier receives all flattened `32 x 5 x 5` features.
Now build a second model that performs **average pooling before the final linear layer**.

## Why do this?
Average pooling reduces the feature map to one value per channel. This strongly reduces the number of parameters in the final classifier.

## Your goal
Implement a model that keeps the same convolutional backbone as before, but adds an average-pooling step so that the final linear layer only receives **32 input features** instead of `32*5*5`.

In [ ]:
class ConvModel2(nn.Module):

    def __init__(self):
        super(ConvModel2, self).__init__()

        # TODO: implement a second convolutional model based on the first one,
        # but reduce the representation before the final classifier using
        # average pooling.
        self.conv_layers = nn.Sequential(
            # ...
        )

        # TODO: define the final classifier.
        # self.linear_layer = ...

    def forward(self, input):
        # TODO: apply the network and return class logits.
        raise NotImplementedError("Implement the ConvModel2 forward pass")

In [ ]:
# TODO: Set up `ConvModel2` for training.
#
# Keep the comparison with the previous models meaningful by choosing a
# consistent training setup.

In [ ]:
# TODO: Train `ConvModel2` and visualize its learning curves.
#
# Afterwards, compare all three models in terms of both performance and model size.

# Compare Number of Parameters

A model with fewer trainable parameters is often easier to store and can be less prone to overfitting.
Here, you will compare how parameter-efficient the convolutional models are relative to the MLP.

## Task
- print the number of trainable parameters for `model_mlp`, `model_conv`, and `model_conv2`,
- compare the numbers,
- explain briefly why convolutional networks usually need fewer parameters for image tasks.

Think especially about the difference between:
- fully connected layers, which connect every input feature to every neuron,
- convolutional layers, which reuse small filters across the entire image.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# TODO: Print the number of trainable parameters of all three models.
#
# Then briefly interpret the difference.

# First Conv Layer Filter Visualization
## What does a convolutional network look for in the raw image?

The first convolutional layer of a trained network often learns simple visual patterns such as:
- edges,
- color contrasts,
- orientation-sensitive filters,
- texture-like structures.

In this part, you will inspect the filters of a pretrained AlexNet model and visualize them as small images.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

### Example: AlexNet pretrained on ImageNet
AlexNet was trained on the large ImageNet dataset (more than 1 million images, 1000 classes).

We will not train AlexNet here. Instead, we will load a pretrained model and inspect the **weights of the first convolutional layer**.

Reference:
https://github.com/pytorch/vision/blob/master/torchvision/models/alexnet.py

In [ ]:
# TODO: Load a pretrained AlexNet model from `torchvision`.
#
# Use it for the visualization exercises below.

In [ ]:
# TODO: Inspect the AlexNet architecture.
#
# Identify the first convolutional layer and note what you can learn from its shape.

In [ ]:
# TODO: Extract the weights of the first convolutional layer
# and convert them into a NumPy array for further processing.

In [ ]:
# TODO: Check the shape of the extracted filter tensor
# and interpret what each dimension represents.

In [ ]:
# TODO: Rearrange the filter tensor into a format that is convenient
# for visualization with `matplotlib`.

In [ ]:
# TODO: Create a copy of the filters for visualization.
#
# Keep the original tensor unchanged.

In [ ]:
# TODO: Normalize each filter independently so the values can be
# displayed as image intensities.

In [ ]:
# TODO: Visualize the first-layer filters in a grid.
#
# After plotting, look for simple patterns such as edges, color transitions,
# or directional structure.